In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [ ]:
import os
import sys

sys.path.append("..")

import matplotlib.cm as cm
import numpy as np
import torch
import wandb
from matplotlib import pyplot as plt
from tqdm import tqdm

from src.distributions import StandardNormalSampler, SwissRollSampler
from src.light_gcot import LightGCOT

In [ ]:
torch.set_default_tensor_type(torch.DoubleTensor)

## 2. Config

In [26]:
X_DIM = 2
Y_DIM = 2
assert X_DIM > 1
# assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 500
M_POTENTIALS = 4
INIT_BY_SAMPLES = False
A_DIAGONAL_INIT = 0.1
IS_B_DIAGONAL = True

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

EPSILON = 0.1

D_LR = 3e-4 # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float('inf')

PLOT_EVERY = 500
MAX_STEPS = 20000
CONTINUE = -1

In [27]:
torch.manual_seed(OUTPUT_SEED)

EPS = EPSILON

In [28]:
EXP_COST = "uniform_on_circle_plus_x"
EXP_COST_INCLUDED = False
EXP_META_INFO = "_R=2_D=diag(0.1, 10)"
EXP_NAME = (
    f"LightGCOT_Swiss_Roll_EPSILON_{EPSILON}_MAX_STEPS_{MAX_STEPS}_N_{N_POTENTIALS}_M_{M_POTENTIALS}_with_{EXP_COST}_cost_included_{EXP_COST_INCLUDED}"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    INIT_BY_SAMPLES=INIT_BY_SAMPLES,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    IS_B_DIAGONAL=IS_B_DIAGONAL,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## 3. Create samplers

In [29]:
X_sampler = StandardNormalSampler(dim=X_DIM, device="cpu")
Y_sampler = SwissRollSampler(dim=Y_DIM, device="cpu")

## 4. Model initialization

In [30]:
D = LightGCOT(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    m_potentials=M_POTENTIALS,
    epsilon=EPSILON,
    sampling_batch_size=SAMPLING_BATCH_SIZE,
    A_diagonal_init=A_DIAGONAL_INIT,
    is_B_diagonal=IS_B_DIAGONAL,
    cost_function=EXP_COST,
)

if INIT_BY_SAMPLES:
    D.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

if CONTINUE > -1:
    D_opt.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_{OUTPUT_SEED}_{CONTINUE}.pt")))

## 5. Model training

In [31]:
NUM_STARTING_POINTS = 6

In [32]:
colors_chosen = cm.rainbow(np.linspace(0.1, 0.9, NUM_STARTING_POINTS))

In [33]:
def logfig(model_: LightGCOT, num_sample: int = 1024) -> str:
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.75), dpi=200)

    for ax in axes:
        ax.grid(zorder=-20)

    x_samples = X_sampler.sample(num_sample)
    y_samples = Y_sampler.sample(num_sample)
    tr_samples = torch.tensor([[-2.0, -1.0], [1.75, -1.75], [-1.5, 1.5], [2, 2], [2.5, 0], [0.0, 0.0]])

    tr_samples = tr_samples[None].repeat(3, 1, 1).reshape(3 * NUM_STARTING_POINTS, 2)

    axes[0].scatter(
        x_samples[:, 0], x_samples[:, 1], alpha=0.3, c="g", s=32, edgecolors="black", label=r"Input distirubtion $p_0$"
    )
    axes[0].scatter(
        y_samples[:, 0], y_samples[:, 1], c="orange", s=32, edgecolors="black", label=r"Target distribution $p_1$"
    )

    for ax, model in zip(axes[1:], [model_]):
        y_pred = model(x_samples)

        ax.scatter(y_pred[:, 0], y_pred[:, 1], c="yellow", s=32, edgecolors="black", label="Fitted distribution", zorder=1)

        tr_samples_pred = model(tr_samples)
        for i in range(3):
            for j, (point, point_pred) in enumerate(
                zip(
                    tr_samples[i * NUM_STARTING_POINTS : (i + 1) * NUM_STARTING_POINTS],
                    tr_samples_pred[i * NUM_STARTING_POINTS : (i + 1) * NUM_STARTING_POINTS],
                )
            ):
                ax.scatter(
                    point[0],
                    point[1],
                    color=colors_chosen[j],
                    s=32,
                    # label=r"Trajectory start ($x \sim p_0$)",
                    zorder=3,
                    edgecolors="black",
                )
                ax.scatter(
                    point_pred[0],
                    point_pred[1],
                    color=colors_chosen[j],
                    s=32,
                    # label=r"Trajectory end ($x \sim p_1$)",
                    zorder=3,
                    edgecolors="black",
                )
                ax.arrow(
                    point[0],
                    point[1],
                    point_pred[0] - point[0],
                    point_pred[1] - point[1],
                )


    for ax in axes:
        ax.set_xlim([-2.5, 2.5])
        ax.set_ylim([-2.5, 2.5])
        ax.legend(loc="lower left")

    fig.tight_layout(pad=0.1)
    plt.close(fig)

    wandb.log({"Image": wandb.Image(fig)})

In [37]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(CONTINUE + 1, 1 * MAX_STEPS)):
    # training loop
    D_opt.zero_grad()

    X, Y = X_sampler.sample(BATCH_SIZE), Y_sampler.sample(BATCH_SIZE)

    b_m = D.compute_b_m(X)  # [bs x M x y_dim]
    B_m = D.compute_B_m(X)  # [bs x M x y_dim]
    log_v_m = D.compute_log_v_m(X)  # [bs x M]
    
    f_c = D.compute_dual_potential(b_m, B_m, log_v_m)
    f = D.compute_primal_potential(Y)

    D_loss = -(f_c + f).mean()
    D_loss.backward()
    D_gradient_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
    D_opt.step()

    wandb.log({f"D gradient norm": D_gradient_norm.item()}, step=step)
    wandb.log({f"D_loss": D_loss.item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(D.A_diagonal_matrix)}, step=step)
    wandb.log({f"lam_min(B_m)": torch.min(D.B_m_matrix)}, step=step)

    if step % PLOT_EVERY == 0:
        logfig(D)

torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D.pt"))
torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt.pt"))

wandb.finish()

100%|██████████| 20000/20000 [05:28<00:00, 60.83it/s]


D gradient norm,▄▂▂▃▃▃▁▁▃▃▄▄▂▂▅▃▃▂▄▂▃▃▃▂▁▆▂▃▄▂█▃▃▂▂▃▂▄▃▃
D_loss,▃▅▂▇▆▇▃▂▇█▇█▁▂▆▅▄▄▅▃▄▃▁▇▂▆▂▄▇▄▅▄▆▄▇▃▁▂▅▅
lam_min(A_n),▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
lam_min(B_m),▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
D gradient norm,0.31914
D_loss,-1.96967
lam_min(A_n),0.72506
lam_min(B_m),1.1


In [ ]:
wandb.finish()

In [ ]:
torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D.pt"))
torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt.pt"))

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
y_pred = D(X)
    
plt.scatter(y_pred[:, 0], y_pred[:, 1], 
            c="yellow", s=32, edgecolors="black", label = "Fitted distribution", zorder=1)

In [ ]:
D.A_diagonal_matrix[:, 1].shape

In [ ]:
plt.hist(D.A_diagonal_matrix[:, 0].detach().numpy())

In [ ]:
plt.hist(D.a[:, 1].detach().numpy())

In [ ]:
plt.hist(D.a[:, 0].detach().numpy())